# SAE Operational Fingerprint — Math Operations

**Hypothesis**: when a language model computes a specific arithmetic operation it activates a characteristic circuit whose signature is readable in the residual stream. A Sparse Autoencoder (SAE) decomposes that stream into interpretable features; the subset with high Cohen's d vs. a non-math baseline is the **operational fingerprint**.

| | |
|--|--|
| **Baseline** | Neutral non-math ctrl prompts (Cohen's d anchor) |
| **Conditions** | add / sub / mul / div / copy |
| **Holdouts** | H1 per-op compute · H2 per-op cheat · H3 multi-op · H4 multi-op cheat |
| **Analysis** | All formats combined + per format (symbolic / mixed / verbal) |


## 0 · Configuration

In [ ]:
# ═══════════════════════════════════════════════════════
#  Edit this cell — everything else adapts automatically
# ═══════════════════════════════════════════════════════
MODEL_NAME       = "google/gemma-3-1b-pt"  # swap for a larger model
LAYER            = 17         # residual-stream layer to hook (0-indexed)
N_PER_CELL       = 150        # examples per (op, format, bin) → ~10 k train
DEVICE_OVERRIDE  = None       # "cuda" / "mps" / "cpu" or None for auto

SAE_K            = 32         # TopK sparsity
SAE_RATIO        = 8          # d_sae = d_model × SAE_RATIO
SAE_LR           = 3e-4
SAE_EPOCHS       = 30
WARMUP_STEPS     = 200
SAE_BATCH        = 512
FP_THRESHOLD     = 0.5        # Cohen's d cut-off
AUX_W            = 1 / 32
DEAD_THR         = 1e-4

N_CTRL           = 1500       # non-math control prompts (fingerprint baseline)
HOLDOUT_FRAC     = 0.20
CHEAT_SAMPLE     = 300        # cheat records per op for H2

OPS_EVAL  = ["add", "sub", "mul", "div"]
FORMATS   = ["symbolic", "mixed", "verbal"]
COLORS    = {"add":"steelblue","sub":"seagreen","mul":"darkorange",
             "div":"mediumpurple","copy":"tomato","ctrl":"gray"}


## 1 · Setup

In [ ]:
import subprocess, sys
def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
for mod, pkg in [("torch","torch"),("transformers","transformers accelerate"),
                 ("scipy","scipy"),("sklearn","scikit-learn"),
                 ("tqdm","tqdm"),("seaborn","seaborn")]:
    try: __import__(mod)
    except ImportError: pip(*pkg.split())

import re, json, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from transformers import AutoModelForCausalLM, AutoTokenizer
from scipy.stats import spearmanr
from numpy.linalg import norm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from tqdm.auto import tqdm

if DEVICE_OVERRIDE:
    DEVICE = torch.device(DEVICE_OVERRIDE)
elif torch.cuda.is_available():    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available(): DEVICE = torch.device("mps")
else: DEVICE = torch.device("cpu")

OUT_DIR = Path("results"); OUT_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid", font_scale=0.9)
%matplotlib inline
print(f"Device: {DEVICE}")


## 2 · Data generation (self-contained)

In [ ]:

_ONES = ["","one","two","three","four","five","six","seven","eight","nine",
         "ten","eleven","twelve","thirteen","fourteen","fifteen","sixteen",
         "seventeen","eighteen","nineteen"]
_TENS = ["","","twenty","thirty","forty","fifty","sixty","seventy","eighty","ninety"]

def _num_to_words(n):
    if n==0:   return "zero"
    if n<20:   return _ONES[n]
    if n<100:  t,o=divmod(n,10); return _TENS[t]+("-"+_ONES[o] if o else "")
    if n<1000: h,r=divmod(n,100); return _ONES[h]+" hundred"+(" "+_num_to_words(r) if r else "")
    if n<10000: th,r=divmod(n,1000); return _ONES[th]+" thousand"+(" "+_num_to_words(r) if r else "")
    th,r=divmod(n,1000); return _num_to_words(th)+" thousand"+(" "+_num_to_words(r) if r else "")

def _safe_words(n):
    try:    return _num_to_words(n) if n is not None else ""
    except: return str(n)

BINS        = {"1d":(1,9),"2d":(10,99),"3d":(100,999),"4d":(1000,9999),"5d":(10000,99999)}
VERBAL_BINS = {"1d","2d","3d"}

TEMPLATES = {
    "add":[("symbolic","{a}+{b}=","{a}+{b}={c}. So {a}+{b}=",None),
           ("mixed","What is {a} plus {b}?",
            "What is {a} plus {b}? It equals {c}. So what is {a} plus {b}?",
            "The sum is {c}. The sum is"),
           ("verbal","What is {wa} plus {wb}?",
            "{wa} plus {wb} is {wc}. So {wa} plus {wb} is",
            "The answer is {wc}. The answer is")],
    "sub":[("symbolic","{a}-{b}=","{a}-{b}={c}. So {a}-{b}=",None),
           ("mixed","What is {a} minus {b}?",
            "What is {a} minus {b}? It equals {c}. So what is {a} minus {b}?",
            "The result is {c}. The result is"),
           ("verbal","What is {wa} minus {wb}?",
            "{wa} minus {wb} is {wc}. So {wa} minus {wb} is",
            "The answer is {wc}. The answer is")],
    "mul":[("symbolic","{a}*{b}=","{a}*{b}={c}. So {a}*{b}=",None),
           ("mixed","What is {a} times {b}?",
            "What is {a} times {b}? It equals {c}. So what is {a} times {b}?",
            "The product is {c}. The product is"),
           ("verbal","What is {wa} times {wb}?",
            "{wa} times {wb} is {wc}. So {wa} times {wb} is",
            "The answer is {wc}. The answer is")],
    "div":[("symbolic","{a}/{b}=","{a}/{b}={c}. So {a}/{b}=",None),
           ("mixed","What is {a} divided by {b}?",
            "What is {a} divided by {b}? It equals {c}. So what is {a} divided by {b}?",
            "The quotient is {c}. The quotient is"),
           ("verbal","What is {wa} divided by {wb}?",
            "{wa} divided by {wb} is {wc}. So {wa} divided by {wb} is",
            "The answer is {wc}. The answer is")],
}
COPY_TEMPLATES=[("symbolic","Repeat: {c}."),
                ("mixed","The number is {c}. Write the number:"),
                ("verbal","The answer is {wc}. The answer is")]
_CTRL_TEMPLATES=[
    ("The capital of {country} is",{"country":["France","Germany","Japan","Brazil","Canada","Italy"]}),
    ("The color of the sky is",{}),
    ("She walked into the {room} and",{"room":["kitchen","library","office","garden","basement"]}),
    ("The largest {animal} in the world is",{"animal":["mammal","reptile","bird","fish"]}),
    ("He picked up the {obj} and",{"obj":["book","pen","phone","key","bag","bottle"]}),
    ("Scientists recently discovered that",{}),
    ("The best way to learn a language is",{}),
    ("After the storm the {place} was",{"place":["forest","street","beach","market"]}),
    ("In the morning she always",{}),
    ("The old {thing} had been there for years",{"thing":["building","bridge","tree","clock"]}),
]

def _fill(tmpl,a,b,c):
    wa,wb,wc=_safe_words(a),_safe_words(b),_safe_words(c)
    return (tmpl.replace("{a}",str(a) if a is not None else "")
                .replace("{b}",str(b) if b is not None else "")
                .replace("{c}",str(c) if c is not None else "")
                .replace("{wa}",wa).replace("{wb}",wb).replace("{wc}",wc))

def _sample_pair(op,bn,rng):
    lo,hi=BINS[bn]
    for _ in range(100):
        a,b=rng.randint(lo,hi),rng.randint(lo,hi)
        if op=="add": return a,b,a+b
        if op=="sub": a,b=max(a,b),min(a,b); return a,b,a-b
        if op=="mul":
            if a*b<100_000_000: return a,b,a*b
        if op=="div":
            if b!=0 and a%b==0 and a//b>0: return a,b,a//b
    return None

def make_dataset(n_per_cell=150,ops=None,seed=42):
    if ops is None: ops=["add","sub","mul","div"]
    rng=random.Random(seed); data=[]
    for op in ops:
        for bn in BINS:
            for fmt,compute_t,cheat_t,copy_t in TEMPLATES[op]:
                if fmt=="verbal" and bn not in VERBAL_BINS: continue
                gen=att=0
                while gen<n_per_cell and att<n_per_cell*10:
                    att+=1; pair=_sample_pair(op,bn,rng)
                    if pair is None: continue
                    a,b,c=pair; base=dict(op=op,bin=bn,fmt=fmt,a=a,b=b,expected=c)
                    data.append({**base,"variant":"compute","prompt":_fill(compute_t,a,b,c)})
                    data.append({**base,"variant":"cheat",  "prompt":_fill(cheat_t,a,b,c)})
                    if copy_t:
                        data.append({**base,"variant":"copy_op","prompt":_fill(copy_t,a,b,c)})
                    gen+=1
    for bn in BINS:
        lo,hi=BINS[bn]
        for fmt,copy_t in COPY_TEMPLATES:
            if fmt=="verbal" and bn not in VERBAL_BINS: continue
            for _ in range(n_per_cell):
                c=rng.randint(lo,hi)
                wc=_num_to_words(c) if bn in VERBAL_BINS else ""
                prompt=copy_t.replace("{c}",str(c)).replace("{wc}",wc)
                data.append(dict(op="copy",variant="copy",fmt=fmt,bin=bn,
                                 a=None,b=None,expected=c,prompt=prompt))
    rng.shuffle(data); return data

def make_ctrl_data(n,seed=7):
    rng=random.Random(seed); data=[]
    for _ in range(n):
        tmpl,fills=rng.choice(_CTRL_TEMPLATES); prompt=tmpl
        for k,opts in fills.items(): prompt=prompt.replace("{"+k+"}",rng.choice(opts))
        data.append(dict(op="ctrl",variant="ctrl",fmt="none",bin="none",
                         a=None,b=None,expected=None,prompt=prompt))
    return data

def split_dataset(data,holdout_frac=0.20,seed=0):
    rng=random.Random(seed)
    cheat=[r for r in data if r["variant"]=="cheat"]
    non_cheat=[r for r in data if r["variant"]!="cheat"]
    groups=defaultdict(list)
    for rec in non_cheat:
        groups[(rec["op"],rec["variant"],rec["fmt"],rec["bin"])].append(rec)
    train,hold=[],[]
    for grp in groups.values():
        rng.shuffle(grp); n=max(1,int(len(grp)*holdout_frac))
        hold.extend(grp[:n]); train.extend(grp[n:])
    rng.shuffle(train); rng.shuffle(hold); rng.shuffle(cheat)
    return train,hold,cheat

print("Data-generation functions ready.")


In [ ]:
all_data = make_dataset(n_per_cell=N_PER_CELL, ops=OPS_EVAL)
train, hold_per_op, hold_cheat = split_dataset(all_data, holdout_frac=HOLDOUT_FRAC)
ctrl_data    = make_ctrl_data(N_CTRL)
train_corpus = [r for r in train if r["variant"] in ("compute","copy")] + ctrl_data

print(f"Train corpus : {len(train_corpus)}")
print(f"  math       : {sum(1 for r in train_corpus if r['op'] in OPS_EVAL)}")
print(f"  copy       : {sum(1 for r in train_corpus if r['op']=='copy')}")
print(f"  ctrl       : {sum(1 for r in train_corpus if r['op']=='ctrl')}")
print(f"H1 per-op    : {len(hold_per_op)}")
print(f"H2 cheat     : {len(hold_cheat)}")

# Dataset summary heatmap
fig, axes = plt.subplots(1, 3, figsize=(14, 3), sharey=True)
for ax, fmt in zip(axes, FORMATS):
    mat = np.zeros((len(OPS_EVAL), len(BINS)))
    for i,op in enumerate(OPS_EVAL):
        for j,bn in enumerate(BINS):
            mat[i,j] = sum(1 for r in train_corpus
                           if r["op"]==op and r.get("fmt")==fmt
                           and r.get("bin")==bn and r["variant"]=="compute")
    sns.heatmap(mat, annot=True, fmt=".0f", ax=ax, cmap="Blues",
                xticklabels=list(BINS), yticklabels=OPS_EVAL, cbar=(fmt==FORMATS[-1]))
    ax.set_title(f"Format: {fmt}"); ax.set_xlabel("bin")
fig.suptitle("Training examples per (op, bin) — compute variant", y=1.02)
plt.tight_layout(); plt.show()


## 3 · Model

In [ ]:
DTYPE = torch.float16 if str(DEVICE)=="cuda" else torch.float32
print(f"Loading {MODEL_NAME}  dtype={DTYPE}  device={DEVICE} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE)
model.eval().to(DEVICE)
D_MODEL = model.config.hidden_size
D_SAE   = D_MODEL * SAE_RATIO
N_LAYERS = model.config.num_hidden_layers
print(f"d_model={D_MODEL}  d_sae={D_SAE}  layers={N_LAYERS}")
inp = tokenizer("3+5=", return_tensors="pt").to(DEVICE)
with torch.no_grad():
    lg = model(**inp).logits[0,-1].float().cpu().topk(3)
preds = [(tokenizer.decode([i]).strip(), round(v.item(),1))
         for i,v in zip(lg.indices,lg.values)]
print(f"Sanity  3+5= → {preds}")


## 4 · Accuracy (H1 holdout)

In [ ]:
_buf = []
def _pre_hook(module, args):
    hidden = args[0] if isinstance(args[0], torch.Tensor) else args[0][0]
    _buf.append(hidden[:,-1,:].detach().float().cpu())
hook_handle = model.model.layers[LAYER].register_forward_pre_hook(_pre_hook)

@torch.no_grad()
def collect_acts(records, desc="acts"):
    _buf.clear()
    for rec in tqdm(records, desc=desc, leave=False):
        inp = tokenizer(rec["prompt"], return_tensors="pt").to(DEVICE)
        model(**inp)
    return torch.cat(_buf, dim=0)

@torch.no_grad()
def measure_accuracy(records):
    for rec in tqdm(records, desc="acc", leave=False):
        if rec.get("expected") is None: rec["correct"]=None; continue
        inp = tokenizer(rec["prompt"], return_tensors="pt").to(DEVICE)
        out = model.generate(inp["input_ids"], max_new_tokens=6, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
        decoded = tokenizer.decode(out[0, inp["input_ids"].shape[1]:]).strip()
        m = re.match(r"\d+", decoded)
        rec["correct"] = (int(m.group())==rec["expected"]) if m else False


In [ ]:
print("Measuring accuracy on H1 holdout …")
measure_accuracy(hold_per_op)

acc = {}
for op in OPS_EVAL:
    for fmt in FORMATS:
        for bn in BINS:
            grp=[r for r in hold_per_op if r["op"]==op and r["fmt"]==fmt
                 and r["bin"]==bn and r["variant"]=="compute"]
            if grp: acc[(op,fmt,bn)] = float(np.mean([r["correct"] for r in grp]))
print(f"Accuracy cells collected: {len(acc)}")


In [ ]:
# Accuracy heatmaps per format
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
for ax, fmt in zip(axes, FORMATS):
    mat = np.full((len(OPS_EVAL),len(BINS)),np.nan)
    for i,op in enumerate(OPS_EVAL):
        for j,bn in enumerate(BINS):
            v=acc.get((op,fmt,bn)); mat[i,j]=v if v is not None else np.nan
    sns.heatmap(mat, annot=True, fmt=".0%", ax=ax, cmap="RdYlGn", vmin=0, vmax=1,
                xticklabels=list(BINS), yticklabels=OPS_EVAL, cbar=(fmt==FORMATS[-1]), linewidths=0.5)
    ax.set_title(fmt, fontweight="bold"); ax.set_xlabel("Operand bin")
fig.suptitle(f"Accuracy — {MODEL_NAME}  layer {LAYER}", y=1.02)
plt.tight_layout(); plt.savefig(OUT_DIR/"accuracy.png",dpi=150,bbox_inches="tight"); plt.show()

# Collapsed bar chart
fig, ax = plt.subplots(figsize=(8,3.5))
x=np.arange(len(BINS)); w=0.2
for k,op in enumerate(OPS_EVAL):
    vals=[np.nanmean([acc.get((op,fmt,bn),np.nan) for fmt in FORMATS]) for bn in BINS]
    ax.bar(x+k*w, vals, w, label=op, color=COLORS[op], alpha=0.85)
ax.set_xticks(x+1.5*w); ax.set_xticklabels(list(BINS))
ax.set_ylabel("Accuracy (avg across formats)"); ax.set_xlabel("Operand bin")
ax.set_title("Model accuracy by operand magnitude"); ax.legend(); ax.set_ylim(0,1)
plt.tight_layout(); plt.show()


## 5 · Activation collection

In [ ]:
print(f"Collecting activations for {len(train_corpus)} records …")
train_acts_raw = collect_acts(train_corpus, desc="train")
n_nan = torch.isnan(train_acts_raw).sum().item()
if n_nan: print(f"  WARNING: {n_nan} NaN → replacing with 0")
train_acts_raw = torch.nan_to_num(train_acts_raw, nan=0.0, posinf=0.0, neginf=0.0)
act_mean = train_acts_raw.mean(dim=0,keepdim=True)
act_std  = train_acts_raw.std(dim=0, keepdim=True).clamp(min=1e-6)
train_acts = (train_acts_raw - act_mean) / act_std
print(f"  Shape: {train_acts.shape}  mean={train_acts.mean():.4f}  std={train_acts.std():.4f}")


## 6 · SAE training

In [ ]:
class TopKSAE(nn.Module):
    def __init__(self,d_in,d_sae,k):
        super().__init__(); self.k=k
        self.pre_bias=nn.Parameter(torch.zeros(d_in))
        self.W_enc=nn.Parameter(torch.empty(d_in,d_sae)); self.b_enc=nn.Parameter(torch.zeros(d_sae))
        self.W_dec=nn.Parameter(torch.empty(d_sae,d_in)); self.b_dec=nn.Parameter(torch.zeros(d_in))
        nn.init.kaiming_uniform_(self.W_enc); nn.init.kaiming_uniform_(self.W_dec)
        with torch.no_grad(): self.W_dec.data=nn.functional.normalize(self.W_dec.data,dim=1)
    def encode(self,x):
        pre=(x-self.pre_bias)@self.W_enc+self.b_enc
        vals,idx=pre.topk(self.k,dim=-1)
        acts=torch.zeros_like(pre); acts.scatter_(-1,idx,vals.clamp(min=0)); return acts
    def decode(self,acts): return acts@self.W_dec+self.b_dec
    def forward(self,x): acts=self.encode(x); return acts,self.decode(acts)
    @torch.no_grad()
    def normalise_decoder(self): self.W_dec.data=nn.functional.normalize(self.W_dec.data,dim=1)

sae=TopKSAE(D_MODEL,D_SAE,SAE_K)
print(f"SAE: {D_MODEL}→{D_SAE}  k={SAE_K}  params={sum(p.numel() for p in sae.parameters()):,}")


In [ ]:
optimizer  = torch.optim.Adam(sae.parameters(),lr=SAE_LR)
loader     = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(train_acts), batch_size=SAE_BATCH, shuffle=True)
feat_usage = torch.zeros(D_SAE); history=[]; step=0

for epoch in tqdm(range(SAE_EPOCHS), desc="SAE epochs"):
    e_mse=0.0
    for (x,) in loader:
        lr=SAE_LR*min(1.0,step/max(1,WARMUP_STEPS))
        for pg in optimizer.param_groups: pg["lr"]=lr
        step+=1
        acts,x_hat=sae(x)
        with torch.no_grad(): feat_usage=0.99*feat_usage+0.01*(acts>0).float().mean(0)
        mse=(x-x_hat).pow(2).mean()
        dead=(feat_usage<DEAD_THR).float()
        aux=((x-x_hat).detach()-(acts*dead)@sae.W_dec).pow(2).mean() if dead.sum()>0 else torch.tensor(0.)
        optimizer.zero_grad(); (mse+AUX_W*aux).backward()
        nn.utils.clip_grad_norm_(sae.parameters(),1.0)
        optimizer.step(); sae.normalise_decoder(); e_mse+=mse.item()
    n_dead=(feat_usage<DEAD_THR).sum().item(); mean_mse=e_mse/len(loader)
    if not torch.isfinite(torch.tensor(mean_mse)): print("NaN — stopping"); break
    history.append(dict(epoch=epoch+1,mse=mean_mse,dead=n_dead))

sae.eval()
with torch.no_grad():
    s=train_acts[:1000]; var_expl=1-(s-sae(s)[1]).pow(2).sum()/s.pow(2).sum()
n_live=(feat_usage>DEAD_THR).sum().item()
print(f"Variance explained: {var_expl.item():.2%}   Live: {n_live}/{D_SAE}")


In [ ]:
fig,(a1,a2)=plt.subplots(1,2,figsize=(11,3.5))
epochs=[h["epoch"] for h in history]
a1.plot(epochs,[h["mse"] for h in history],marker="o",ms=3,color="steelblue")
a1.set(title="SAE reconstruction MSE",xlabel="Epoch",ylabel="MSE")
a2.plot(epochs,[h["dead"] for h in history],marker="o",ms=3,color="tomato")
a2.axhline(D_SAE*0.5,ls="--",color="gray",alpha=0.5,label="50% dead")
a2.set(title=f"Dead features (/{D_SAE})",xlabel="Epoch"); a2.legend()
plt.suptitle(f"SAE training — d_sae={D_SAE}  k={SAE_K}",y=1.01)
plt.tight_layout(); plt.savefig(OUT_DIR/"sae_training.png",dpi=150,bbox_inches="tight"); plt.show()


## 7 · Fingerprints & format invariance

Baseline for Cohen's d = **ctrl** (neutral non-math text). Copy is a labelled condition, not the baseline.

In [ ]:
@torch.no_grad()
def to_features(acts):
    return np.concatenate([sae.encode(acts[i:i+512]).numpy()
                           for i in range(0,len(acts),512)],axis=0)

all_feats_np = np.nan_to_num(to_features(train_acts),nan=0.0)
train_recs   = train_corpus

def get_slice(op=None,fmt=None):
    mask=np.ones(len(train_recs),dtype=bool)
    if op:  mask&=np.array([r["op"] ==op  for r in train_recs])
    if fmt: mask&=np.array([r["fmt"]==fmt for r in train_recs])
    return all_feats_np[mask]

def compute_fps(fd):
    if "ctrl" not in fd or len(fd["ctrl"])==0: return {},{}
    bm=fd["ctrl"].mean(0); bs=fd["ctrl"].std(0)+1e-8
    fps,effs={},{}
    for op in OPS_EVAL:
        if op not in fd or len(fd[op])==0: continue
        d=(fd[op].mean(0)-bm)/np.sqrt((bs**2+fd[op].std(0)**2+1e-8)/2)
        effs[op]=d; fps[op]=d>FP_THRESHOLD
    return fps,effs

def containment(a,b): return float((a&b).sum())/(a.sum()+1e-8)
def cos(a,b): na,nb=norm(a),norm(b); return float(np.dot(a,b)/(na*nb)) if na>1e-8 and nb>1e-8 else 0.

feats_all={op:get_slice(op=op) for op in OPS_EVAL}
feats_all["copy"]=get_slice(op="copy"); feats_all["ctrl"]=get_slice(op="ctrl")
feats_by_fmt={fmt:{**{op:get_slice(op=op,fmt=fmt) for op in OPS_EVAL},
                   "copy":get_slice(op="copy",fmt=fmt),"ctrl":get_slice(op="ctrl")}
              for fmt in FORMATS}

fps_all,effs_all=compute_fps(feats_all)
fps_by_fmt={fmt:compute_fps(feats_by_fmt[fmt])[0] for fmt in FORMATS}

print(f"Fingerprint sizes (d>{FP_THRESHOLD} vs ctrl):")
print(f"  {'op':5s} {'all':>6s}",*[f"{f:>10s}" for f in FORMATS])
for op in OPS_EVAL:
    n_all=int(fps_all.get(op,np.zeros(D_SAE,bool)).sum())
    by_f=[int(fps_by_fmt[f].get(op,np.zeros(D_SAE,bool)).sum()) for f in FORMATS]
    print(f"  {op:5s} {n_all:>6d}",*[f"{n:>10d}" for n in by_f])


In [ ]:
# Compute Jaccard across formats
jaccards={}
for op in OPS_EVAL:
    fps_f=[fps_by_fmt[f].get(op,np.zeros(D_SAE,bool)) for f in FORMATS]
    union=np.array(fps_f).any(0); inter=np.array(fps_f).all(0)
    jaccards[op]=float(inter.sum())/(union.sum()+1e-8)

fig,axes=plt.subplots(1,3,figsize=(15,4))

# Panel 1: fingerprint size grouped bar
ax=axes[0]; x=np.arange(len(OPS_EVAL)); w=0.18
sc={"all":"#444","symbolic":"steelblue","mixed":"darkorange","verbal":"seagreen"}
for k,sl in enumerate(["all"]+FORMATS):
    fps_sl=fps_all if sl=="all" else fps_by_fmt[sl]
    vals=[int(fps_sl.get(op,np.zeros(D_SAE,bool)).sum()) for op in OPS_EVAL]
    ax.bar(x+k*w,vals,w,label=sl,color=sc[sl],alpha=0.85)
ax.set_xticks(x+1.5*w); ax.set_xticklabels(OPS_EVAL)
ax.set_ylabel("# features"); ax.set_title("Fingerprint sizes"); ax.legend(fontsize=8)

# Panel 2: Jaccard invariance
ax2=axes[1]
bars=ax2.bar(OPS_EVAL,[jaccards[op] for op in OPS_EVAL],
             color=[COLORS[op] for op in OPS_EVAL],alpha=0.85)
ax2.set_ylim(0,1); ax2.set_ylabel("Jaccard"); ax2.axhline(0.5,ls="--",color="gray",alpha=0.5)
ax2.set_title("Format invariance (Jaccard sym/mix/verb)")
for b,op in zip(bars,OPS_EVAL):
    ax2.text(b.get_x()+b.get_width()/2,b.get_height()+0.02,f"{jaccards[op]:.2f}",ha="center",fontsize=9)

# Panel 3: containment matrix (all formats)
ax3=axes[2]
mat=np.zeros((len(OPS_EVAL),len(OPS_EVAL)))
for i,a in enumerate(OPS_EVAL):
    for j,b in enumerate(OPS_EVAL):
        if a in fps_all and b in fps_all: mat[i,j]=containment(fps_all[a],fps_all[b])
sns.heatmap(mat,annot=True,fmt=".2f",ax=ax3,cmap="Blues",vmin=0,vmax=1,
            xticklabels=OPS_EVAL,yticklabels=OPS_EVAL,linewidths=0.5)
ax3.set_title("Containment [row ⊆ col]"); ax3.set_xlabel("col"); ax3.set_ylabel("row")

plt.suptitle("Format invariance & containment",y=1.02)
plt.tight_layout(); plt.savefig(OUT_DIR/"fingerprints.png",dpi=150,bbox_inches="tight"); plt.show()


In [ ]:
# Containment per format slice (4 panels)
fig,axes=plt.subplots(1,4,figsize=(16,3.5))
for ax,(sl,fps_sl) in zip(axes,[("all",fps_all)]+[(f,fps_by_fmt[f]) for f in FORMATS]):
    mat=np.zeros((len(OPS_EVAL),len(OPS_EVAL)))
    for i,a in enumerate(OPS_EVAL):
        for j,b in enumerate(OPS_EVAL):
            if a in fps_sl and b in fps_sl: mat[i,j]=containment(fps_sl[a],fps_sl[b])
    sns.heatmap(mat,annot=True,fmt=".2f",ax=ax,cmap="Blues",vmin=0,vmax=1,
                xticklabels=OPS_EVAL,yticklabels=OPS_EVAL,linewidths=0.5,cbar=(sl==FORMATS[-1]))
    ax.set_title(sl,fontweight="bold"); ax.set_xlabel("col op")
plt.suptitle("Containment per format slice",y=1.02)
plt.tight_layout(); plt.savefig(OUT_DIR/"containment.png",dpi=150,bbox_inches="tight"); plt.show()


In [ ]:
# Linear AUC: math ops vs ctrl baseline
auc_results={}
for sl,fd in [("all",feats_all)]+[(f,feats_by_fmt[f]) for f in FORMATS]:
    ops_p=[op for op in OPS_EVAL if op in fd and len(fd[op])>=10]
    if not ops_p or len(fd.get("ctrl",[]))<10: continue
    X=np.concatenate([fd[op] for op in ops_p]+[fd["ctrl"]])
    y=np.array([1]*sum(len(fd[op]) for op in ops_p)+[0]*len(fd["ctrl"]))
    aucs=cross_val_score(LogisticRegression(max_iter=500,C=0.1),X,y,
                         cv=min(5,int(y.sum())),scoring="roc_auc")
    auc_results[sl]=(aucs.mean(),aucs.std())
    print(f"  {sl:10s}  AUC={aucs.mean():.3f} ± {aucs.std():.3f}")


## 8 · PCA visualisation

In [ ]:
SUB=250; np.random.seed(0)
ALL_PLOT=OPS_EVAL+["copy","ctrl"]
OP_COLORS={**COLORS,"ctrl":"gray"}

fig=plt.figure(figsize=(18,4.5))
gs=gridspec.GridSpec(1,4,figure=fig,wspace=0.35)

def pca_plot(ax,fd,title):
    ops_p=[op for op in ALL_PLOT if op in fd and len(fd[op])>=5]
    sub_f={op:fd[op][np.random.choice(len(fd[op]),min(SUB,len(fd[op])),replace=False)] for op in ops_p}
    X=np.nan_to_num(np.concatenate(list(sub_f.values())),nan=0.0)
    lbls=sum([[op]*len(sub_f[op]) for op in ops_p],[])
    if X.shape[0]<5: ax.set_title(title+"\n(no data)"); return
    pcs=PCA(n_components=2).fit_transform(StandardScaler().fit_transform(X))
    for op in ops_p:
        m=np.array(lbls)==op
        ax.scatter(pcs[m,0],pcs[m,1],c=OP_COLORS.get(op,"k"),alpha=0.4,s=10,label=op,rasterized=True)
    ax.legend(fontsize=7,markerscale=2); ax.set_title(title,fontweight="bold")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")

pca_plot(fig.add_subplot(gs[0]),feats_all,"All formats")
for i,fmt in enumerate(FORMATS):
    pca_plot(fig.add_subplot(gs[i+1]),feats_by_fmt[fmt],fmt.capitalize())

plt.suptitle(f"PCA of SAE features — {MODEL_NAME}",y=1.02,fontsize=11)
plt.savefig(OUT_DIR/"pca.png",dpi=150,bbox_inches="tight"); plt.show()


## 9 · Cheat holdout

**Q**: when the answer is hinted (`3+5=8. So 3+5=`), does the fingerprint shift toward *copy* or stay at *compute*?

In [ ]:
print("Collecting cheat activations …")
_buf.clear()
cheat_feats={}
for op in OPS_EVAL:
    recs=[r for r in hold_cheat if r["op"]==op][:CHEAT_SAMPLE]
    if not recs: continue
    acts=collect_acts(recs,desc=f"cheat/{op}")
    acts=torch.nan_to_num(acts,nan=0.0)
    acts=(acts-act_mean)/act_std
    cheat_feats[op]=np.nan_to_num(to_features(acts),nan=0.0)
print("Done.")


In [ ]:
copy_centroid=feats_all["copy"].mean(0)
ctrl_centroid=feats_all["ctrl"].mean(0)

cheat_res={}
for op in OPS_EVAL:
    if op not in cheat_feats: continue
    cc=cheat_feats[op].mean(0); comp_c=feats_all[op].mean(0)
    cheat_res[op]=dict(comp_sim=cos(cc,comp_c),copy_sim=cos(cc,copy_centroid),ctrl_sim=cos(cc,ctrl_centroid))

print(f"  {'op':5s} {'→compute':>10s} {'→copy':>8s} {'→ctrl':>8s}  verdict")
for op,r in cheat_res.items():
    v="→copy" if r["copy_sim"]>r["comp_sim"] else "→compute"
    print(f"  {op:5s} {r['comp_sim']:>10.4f} {r['copy_sim']:>8.4f} {r['ctrl_sim']:>8.4f}  {v}")


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
ops_c=[op for op in OPS_EVAL if op in cheat_res]; x=np.arange(len(ops_c)); w=0.28

ax=axes[0]
ax.bar(x-w/2,[cheat_res[op]["comp_sim"] for op in ops_c],w,label="→ compute",color="steelblue",alpha=0.85)
ax.bar(x+w/2,[cheat_res[op]["copy_sim"] for op in ops_c],w,label="→ copy",color="tomato",alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(ops_c); ax.set_ylim(0,1)
ax.set_ylabel("Cosine similarity"); ax.set_title("Cheat: compute vs copy centroid"); ax.legend()
ax.axhline(0.5,ls="--",color="gray",alpha=0.5)

ax2=axes[1]
overlaps=[]
for op in ops_c:
    if op not in fps_all: overlaps.append(float("nan")); continue
    bm=feats_all["ctrl"].mean(0); bs=feats_all["ctrl"].std(0)+1e-8
    d=(cheat_feats[op].mean(0)-bm)/np.sqrt((bs**2+cheat_feats[op].std(0)**2+1e-8)/2)
    fp_ch=d>FP_THRESHOLD
    overlaps.append(containment(fp_ch,fps_all[op]))

bars=ax2.bar(ops_c,overlaps,color=[COLORS[op] for op in ops_c],alpha=0.85)
ax2.set_ylim(0,1); ax2.set_ylabel("Containment(cheat fp ⊆ compute fp)")
ax2.set_title("How much of cheat fingerprint overlaps compute?")
ax2.axhline(0.5,ls="--",color="gray",alpha=0.4)
for b,v in zip(bars,overlaps):
    ax2.text(b.get_x()+b.get_width()/2,b.get_height()+0.02,
             f"{v:.2f}" if not (isinstance(v,float) and v!=v) else "n/a",ha="center",fontsize=9)

plt.suptitle("Cheat holdout analysis",y=1.01)
plt.tight_layout(); plt.savefig(OUT_DIR/"cheat.png",dpi=150,bbox_inches="tight"); plt.show()


## 10 · Summary dashboard

In [ ]:
fig=plt.figure(figsize=(16,10))
gs=gridspec.GridSpec(2,3,figure=fig,hspace=0.45,wspace=0.35)

# A: accuracy heatmap (pooled)
ax=fig.add_subplot(gs[0,0])
mat=np.full((len(OPS_EVAL),len(BINS)),np.nan)
for i,op in enumerate(OPS_EVAL):
    for j,bn in enumerate(BINS):
        vs=[acc.get((op,fmt,bn),np.nan) for fmt in FORMATS]; v=np.nanmean(vs)
        if not np.isnan(v): mat[i,j]=v
sns.heatmap(mat,annot=True,fmt=".0%",ax=ax,cmap="RdYlGn",vmin=0,vmax=1,
            xticklabels=list(BINS),yticklabels=OPS_EVAL,linewidths=0.5)
ax.set_title("A  Accuracy (pooled formats)",fontweight="bold"); ax.set_xlabel("bin")

# B: SAE MSE curve
ax=fig.add_subplot(gs[0,1])
ax.plot([h["epoch"] for h in history],[h["mse"] for h in history],color="steelblue",marker="o",ms=3)
ax.set(title="B  SAE MSE",xlabel="Epoch",ylabel="MSE")

# C: fingerprint sizes
ax=fig.add_subplot(gs[0,2]); x=np.arange(len(OPS_EVAL)); w=0.18
sc={"all":"#444","symbolic":"steelblue","mixed":"darkorange","verbal":"seagreen"}
for k,sl in enumerate(["all"]+FORMATS):
    fps_sl=fps_all if sl=="all" else fps_by_fmt[sl]
    vals=[int(fps_sl.get(op,np.zeros(D_SAE,bool)).sum()) for op in OPS_EVAL]
    ax.bar(x+k*w,vals,w,label=sl,color=sc[sl],alpha=0.85)
ax.set_xticks(x+1.5*w); ax.set_xticklabels(OPS_EVAL)
ax.set_title("C  Fingerprint sizes",fontweight="bold"); ax.legend(fontsize=7); ax.set_ylabel("# features")

# D: Jaccard
ax=fig.add_subplot(gs[1,0])
ax.bar(OPS_EVAL,[jaccards.get(op,0) for op in OPS_EVAL],
       color=[COLORS[op] for op in OPS_EVAL],alpha=0.85)
ax.set_ylim(0,1); ax.set_title("D  Format invariance (Jaccard)",fontweight="bold")
ax.axhline(0.5,ls="--",color="gray",alpha=0.4)

# E: containment (all)
ax=fig.add_subplot(gs[1,1])
mat=np.zeros((len(OPS_EVAL),len(OPS_EVAL)))
for i,a in enumerate(OPS_EVAL):
    for j,b in enumerate(OPS_EVAL):
        if a in fps_all and b in fps_all: mat[i,j]=containment(fps_all[a],fps_all[b])
sns.heatmap(mat,annot=True,fmt=".2f",ax=ax,cmap="Blues",vmin=0,vmax=1,
            xticklabels=OPS_EVAL,yticklabels=OPS_EVAL,linewidths=0.5)
ax.set_title("E  Containment (all formats)",fontweight="bold")

# F: cheat
ax=fig.add_subplot(gs[1,2])
if cheat_res:
    ops_c=list(cheat_res.keys()); xf=np.arange(len(ops_c))
    ax.bar(xf-w/2,[cheat_res[op]["comp_sim"] for op in ops_c],w,label="→compute",color="steelblue",alpha=0.85)
    ax.bar(xf+w/2,[cheat_res[op]["copy_sim"] for op in ops_c],w,label="→copy",color="tomato",alpha=0.85)
    ax.set_xticks(xf); ax.set_xticklabels(ops_c); ax.set_ylim(0,1)
    ax.set_title("F  Cheat proximity",fontweight="bold"); ax.legend(fontsize=8)

plt.suptitle(f"SAE Fingerprint — {MODEL_NAME}  layer {LAYER}  d_sae={D_SAE}  k={SAE_K}",
             fontsize=12,y=1.01)
plt.savefig(OUT_DIR/"summary.png",dpi=150,bbox_inches="tight"); plt.show()


In [ ]:
print("="*60)
print(f"  Model:          {MODEL_NAME}")
print(f"  Layer:          {LAYER} / {N_LAYERS}")
print(f"  d_model:        {D_MODEL}   d_sae: {D_SAE}   k: {SAE_K}")
print(f"  Train examples: {len(train_acts)}")
print(f"  Variance expl:  {var_expl.item():.2%}")
print(f"  Live features:  {n_live}/{D_SAE}  ({n_live/D_SAE:.1%})")
print()
print(f"  Fingerprints (d>{FP_THRESHOLD} vs ctrl):")
for op in OPS_EVAL:
    n=int(fps_all.get(op,np.zeros(D_SAE,bool)).sum()); j=jaccards.get(op,float("nan"))
    print(f"    {op}: {n:4d} features   Jaccard={j:.3f}")
print()
for sl,(m,s) in auc_results.items():
    print(f"  AUC [{sl:10s}]: {m:.3f} ± {s:.3f}")
print("="*60)

json.dump(dict(
    model=MODEL_NAME,layer=LAYER,n_train=int(len(train_acts)),
    var_explained=float(var_expl.item()),n_live=int(n_live),d_sae=D_SAE,
    fps_all={op:int(fps_all.get(op,np.zeros(D_SAE,bool)).sum()) for op in OPS_EVAL},
    jaccard={op:float(jaccards.get(op,float("nan"))) for op in OPS_EVAL},
    auc={sl:float(m) for sl,(m,s) in auc_results.items()},
    cheat={op:{k:float(v) for k,v in r.items()} for op,r in cheat_res.items()},
),open(OUT_DIR/"results.json","w"),indent=2)
print(f"Saved → {OUT_DIR}/results.json  and  {OUT_DIR}/*.png")
